In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme("talk", "ticks")

In [ ]:
dataset = "eso245_cra_strin_10"#"ESO190_strin_AAL90"#"MS_fMRI"#"boldEso190"#
bins = 9#8#

with open(f"../{dataset}_bin{bins}/globalStats.json") as fp:
    globaStata = {k:np.array(v) for k, v in json.load(fp).items()}
npat = len(globaStata["globaltotalMI"])
print(np.argmax(globaStata["globaltotalMI"]-globaStata["globalgaussMI"]))

In [ ]:
from scipy.stats import pearsonr
folderName = f"../{dataset}_bin{bins}/"
patientN = 187
statsMI = np.load(
    f"{folderName}/patient{patientN:02}_{bins}.npy"
)
statsMI_shadow = np.load(
    f"{folderName}/patient{patientN:02}_{bins}_sha.npy"
)
corr = np.load(
    f"{folderName}/patient{patientN:02}_{bins}_cor.npy"
)
sorter = np.argsort( statsMI[:,1:],1)
prctl = np.array([np.searchsorted(statsMI[i,1:], statsMI[i,0],sorter=sorter[i,:]) for i in range(statsMI.shape[0])])
plt.hist2d(np.abs(corr), prctl)#, bins="auto")
plt.title(pearsonr(np.abs(corr), prctl))
print(globaStata['globalratio95'][patientN], globaStata['globalratio99'][patientN])
plt.show()

In [ ]:
ratio = (globaStata["globaltotalMI"]-globaStata["globalgaussMI"])/globaStata["globalgaussMI"]
order = np.argsort(ratio)
topneg=np.argmax(np.cumsum(ratio[order]<0))

plt.bar(np.arange(0,topneg,3), globaStata["globalgaussMI"][order[0:topneg:3]],2,color="g", linewidth=0, label="Gaussian MI")
plt.bar(np.arange(0,topneg,3), globaStata["globaltotalMI"][order[0:topneg:3]],2,color="r", linewidth=0, label="Total MI")

plt.bar(np.arange(topneg, len(ratio),3), globaStata["globaltotalMI"][order[topneg::3]],2,color="r", linewidth=0)
plt.bar(np.arange(topneg, len(ratio),3), globaStata["globalgaussMI"][order[topneg::3]],2,color="g", linewidth=0)

plt.xlim((-2,npat+2))

plt.xlabel("One in three Healthy Contyrols subsample\n(ordered by Total vs Gaussian MI ratio)")
plt.ylabel("Average mutual information (nats)")
plt.title(f"{dataset} dataset using {bins} bins")
plt.legend(loc=2, fontsize="small")
plt.savefig(f"../hist{dataset}_bin{bins}.pdf", bbox_inches="tight")
plt.show()

diff = (globaStata["globaltotalMI"]-globaStata["globalgaussMI"])
diffr = np.argsort(diff)
tpn = np.argmax(np.cumsum(diff[diffr]<0))
plt.scatter(np.arange(len(diff)),diff[diffr], marker="3")
plt.scatter(topneg,diff[diffr[tpn]], marker="3", color="red")
plt.plot([0,npat],[0,0], ":k")
plt.xlabel("Healthy Controls")
plt.ylabel("Total vs Gaussian MI difference")
plt.title(f"{dataset} dataset using {bins} bins")
plt.show()


plt.scatter(np.arange(len(ratio)),ratio[order], marker="3")
plt.scatter(topneg,ratio[order[topneg]], marker="3", color="red")
plt.plot([0,npat],[0,0], ":k")
plt.xlabel("Healthy Controls")
plt.ylabel("Total vs Gaussian MI relative difference")
plt.title(f"{dataset} dataset using {bins} bins")
plt.show()

plt.hist(diff, bins="auto")#np.arange(-0.01, 0.01, 0.002))
medf = np.median(diff)
meaf = np.mean(diff)
ylim = plt.ylim()
plt.plot([medf,]*2, ylim, ":k", label="Median")
plt.plot([meaf,]*2, ylim, "--k", label="Mean")
plt.ylim(ylim)
plt.legend()
plt.xlabel("Total vs Gaussian MI difference")
plt.title(f"{dataset} dataset using {bins} bins")
plt.show()

plt.hist(ratio, bins=np.arange(-0.05, 0.05, 0.01))
med = np.median(ratio)
mea = np.mean(ratio)
ylim = plt.ylim()
plt.plot([med,]*2, ylim, ":k", label="Median")
plt.plot([mea,]*2, ylim, "--k", label="Mean")
plt.ylim(ylim)
plt.legend()
plt.xlabel("Total vs Gaussian MI relative difference")
plt.title(f"{dataset} dataset using {bins} bins")
plt.show()

In [ ]:
ratioSHA = (globaStata["globaltotalMIshadow"]-globaStata["globalgaussMIshadow"])/globaStata["globalgaussMIshadow"]
orderSHA = np.argsort(ratioSHA)
topnegSHA=np.argmax(np.cumsum(ratioSHA[orderSHA]<0))

plt.bar(np.arange(0,topnegSHA,3), globaStata["globalgaussMIshadow"][orderSHA[0:topnegSHA:3]],2,color="g", linewidth=0, label="Gaussian MI")
plt.bar(np.arange(0,topnegSHA,3), globaStata["globaltotalMIshadow"][orderSHA[0:topnegSHA:3]],2,color="r", linewidth=0, label="Total MI")

plt.bar(np.arange(topnegSHA+1, len(ratioSHA),3), globaStata["globaltotalMIshadow"][orderSHA[topnegSHA+1::3]],2,color="r", linewidth=0)
plt.bar(np.arange(topnegSHA+1, len(ratioSHA),3), globaStata["globalgaussMIshadow"][orderSHA[topnegSHA+1::3]],2,color="g", linewidth=0)

plt.xlim((-2,npat+2))

plt.xlabel("One in three Healthy Controls subsample\n(ordered by Total vs Gaussian MI ratio)")
plt.ylabel("Average mutual information (nats)")
plt.title(f"{dataset} SHADOW dataset using {bins} bins")
plt.legend(loc=1, fontsize="small")
plt.show()

diffSHA = (globaStata["globaltotalMIshadow"]-globaStata["globalgaussMIshadow"])
diffrSHA = np.argsort(diffSHA)
tpnSHA = np.argmax(np.cumsum(diffSHA[diffrSHA]<0))
plt.scatter(np.arange(len(diffSHA)),diffSHA[diffrSHA], marker="3")
plt.scatter(topneg,diffSHA[diffrSHA[tpnSHA]], marker="3", color="red")
plt.plot([0,npat],[0,0], ":k")
plt.xlabel("Healthy Controls")
plt.ylabel("Total vs Gaussian MI difference")
plt.title(f"{dataset} SHADOW dataset using {bins} bins")
plt.show()


plt.scatter(np.arange(len(ratioSHA)),ratioSHA[orderSHA], marker="3")
plt.scatter(topnegSHA,ratioSHA[orderSHA[topnegSHA]], marker="3", color="red")
plt.plot([0,npat],[0,0], ":k")
plt.xlabel("Healthy Controls")
plt.ylabel("Total vs Gaussian MI relative difference")
plt.title(f"{dataset} SHADOW dataset using {bins} bins")
plt.show()

plt.hist(diffSHA, bins="auto")#np.arange(-0.01, 0.01, 0.002))
medf = np.median(diffSHA)
meaf = np.mean(diffSHA)
ylim = plt.ylim()
plt.plot([medf,]*2, ylim, ":k", label="Median")
plt.plot([meaf,]*2, ylim, "--k", label="Mean")
plt.ylim(ylim)
plt.legend()
plt.xlabel("Total vs Gaussian MI difference")
plt.title(f"{dataset} SHADOW dataset using {bins} bins")
plt.show()

plt.hist(ratioSHA, bins=np.arange(-0.05, 0.05, 0.01))
medSHA = np.median(ratioSHA)
meaSHA = np.mean(ratioSHA)
ylim = plt.ylim()
plt.plot([medSHA,]*2, ylim, ":k", label="Median")
plt.plot([meaSHA,]*2, ylim, "--k", label="Mean")
plt.ylim(ylim)
plt.legend()
plt.xlabel("Total vs Gaussian MI relative difference")
plt.title(f"{dataset} SHADOW dataset using {bins} bins")
plt.show()

In [ ]:
from scipy.stats import ks_2samp
from scipy.stats import ttest_1samp
from scipy.stats import ttest_rel
from scipy.stats import kstest
from scipy.stats import binomtest
from scipy.stats import binom

n_pairs = int((116-1)*116/2)
N=0
gn95 = (globaStata['globalratio95']*n_pairs)
for rat in  gn95.astype(int):
    if binomtest(rat, n_pairs, 0.05, 'greater').pvalue < 0.05:
        N += 1
print(f"true\nbinomtest sign: {N}, non sign: {npat-N}")
print("t-test too many sign pairs:", ttest_1samp(gn95, 0.05*n_pairs, alternative="greater"))
print("t-test diff greater than 0:", ttest_1samp(diff, 0, alternative="greater"))
# N=0
# gn05 = (globaStata['globalratio05']*n_pairs)
# for rat in  gn05.astype(int):
#     if binomtest(rat, n_pairs, 0.05, 'less').pvalue < 0.05:
#         N += 1
# print(f"true\nbinomtest sign SMALL: {N}, non sign: {npat-N}")
# print("t-test too many sign pairs SMALL:", ttest_1samp(gn05, 0.05*n_pairs, alternative="greater"))

N=0
gn95s = (globaStata['globalratio95shadow']*n_pairs)
for rat in gn95s.astype(int):
    if binomtest(rat, n_pairs, 0.05, 'greater').pvalue < 0.05:
        N += 1
print(f"shadow\nbinomtest sign: {N}, non sign: {npat-N}")
print("t-test too many sign pairs:", ttest_1samp(gn95s, 0.05*n_pairs, alternative="greater"))
print("t shad diff greater than 0:", ttest_1samp(diffSHA, 0, alternative="greater"))
# N=0
# gn05s = (globaStata['globalratio05shadow']*n_pairs)
# for rat in gn05s.astype(int):
#     if binomtest(rat, n_pairs, 0.05, 'less').pvalue < 0.05:
#         N += 1
# print(f"shadow\nbinomtest sign SMALL: {N}, non sign: {npat-N}")
# print("t-test too many sign pairs SMALL:", ttest_1samp(gn05s, 0.05*n_pairs, alternative="greater"))

print("paired\nsign pairs", ttest_rel(gn95, gn95s,alternative='greater'))
print(f"avg negl info: {np.mean(diff):.3} true, {np.mean(diffSHA):.3} shadow")
print("t test: ", ttest_rel(diff, diffSHA, alternative='greater'))
print("ks ratio", ks_2samp(diff, diffSHA))
# print("paired\nsign pairs SMALL", ttest_rel(gn05, gn05s,alternative='greater'))
print("ks ratio", ks_2samp(ratio, ratioSHA))
print("t ratio", ttest_1samp(ratio, 0))
print("t shad r", ttest_1samp(ratioSHA, 0))
print(ttest_rel(ratio, ratioSHA))
print(ttest_rel(diff, diffSHA))

In [ ]:
N=0
for rat in  globaStata['globalratio05']:
    print(f"{rat:.03}", end='')
    if binomtest(int(rat*n_pairs), n_pairs, 0.05, 'greater').pvalue < 0.05:
        print("*,", end=' ')
        N+=1
    else:
        print(", ", end=' ')
print(f"  ->  {N}\n{np.average(globaStata['globalratio05'])}")
N=0
for rat in  globaStata['globalratio05shadow']:
    print(f"{rat:.03}", end='')
    if binomtest(int(rat*n_pairs), n_pairs, 0.05, 'greater').pvalue < 0.05:
        print("*,", end=' ')
        N+=1
    else:
        print(", ", end=' ')
print(f"  ->  {N}\n{np.average(globaStata['globalratio05shadow'])}")

In [ ]:
import glob
import os

statsReg = {}
for folder in glob.glob("../eso245_cra_strin_*"):
    if os.path.isfile(os.path.join(folder,"globalStats.json")):
        statsMI = np.load(os.path.join(folder,f"patient00_9.npy"))
        pairs = statsMI.shape[0]
        regions = int(0.5+np.sqrt(0.25+2*pairs))
        print(folder, regions, statsMI.shape)
        with open(os.path.join(folder,"globalStats.json")) as fp:
            statsReg[regions] = {k:np.array(v) for k, v in json.load(fp).items()}

In [ ]:
x=sorted(statsReg.keys())
# for pat in range(18):
#     yt = [statsReg[i]["globaltotalMI"][pat] for i in x]
#     yg = [statsReg[i]["globalgaussMI"][pat] for i in x]
#     plt.plot(x, yt, label="total")
#     plt.plot(x, yg, label="gauss")
#     plt.ylabel("MI")
#     plt.xlabel("# of regions")
#     plt.title(f"Patient {pat}")
#     plt.legend()
#     plt.show()
for pat in range(18):
    y = [2*(statsReg[i]["globaltotalMI"][pat]-statsReg[i]["globalgaussMI"][pat])/(statsReg[i]["globaltotalMI"][pat]+statsReg[i]["globalgaussMI"][pat]) for i in x]
    plt.plot(x, y, "+-", label=pat)
plt.ylabel("relative difference in MI")
plt.xlabel("# of regions")
plt.xscale("log")
#plt.legend()
plt.show()
